# RQ0b — Product Image Encoder (optional, Mac-only)

**Run this notebook AFTER `RQ0_data_preparation.ipynb`** and before `RQ2_graph_learning.ipynb`.

## What it does
1. Reads `outputs/prepared/amazon_merged.parquet` to get product `images` URLs.
2. Downloads a thumbnail per product (typical 150–300 kB each) with a timeout + retry.
3. Encodes each image with a pretrained vision model (ResNet-50 from torchvision, or CLIP if you prefer).
4. Writes `outputs/prepared/amazon_product_image_embeddings.parquet` (2048-dim for ResNet-50).
5. RQ2 will pick this file up automatically if present and concatenate it with the text embeddings for a true multimodal fusion.

## Why it's optional / separate
- Downloading ~20k images takes 10-30 minutes depending on your connection (the proposal's §6.3 lists image preprocessing as a step, and this is the minimal honest way to do it).
- This notebook requires internet access; RQ2 works fine without it (text-only initial features).
- Image URLs in Amazon's metadata can 404 or rate-limit — the notebook handles failures gracefully and continues.

## Mac M4 runtime estimate

| Step | Runtime |
|---|---|
| Download 5,000 images (small preset) | ~5 min |
| Download 20,000 images (medium preset) | ~20 min |
| ResNet-50 inference on MPS | ~15 s for 5k, ~60 s for 20k |

Set `N_IMAGES` below to control how many products to encode.


In [1]:
import os, io, json, time, warnings, random
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import os as _os, logging as _lg
_os.environ["TRANSFORMERS_VERBOSITY"]         = "error"
_os.environ["HF_HUB_DISABLE_TELEMETRY"]       = "1"
_os.environ["TOKENIZERS_PARALLELISM"]         = "false"
_os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"]= "1"
_os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"]  = "1"
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")
warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")
for _n in ["transformers","sentence_transformers","huggingface_hub","huggingface_hub.utils","urllib3","filelock"]:
    _lg.getLogger(_n).setLevel(_lg.ERROR)

SEED = 42; random.seed(SEED); np.random.seed(SEED)
PROJECT = Path.cwd()
if PROJECT.name == "notebooks": PROJECT = PROJECT.parent
PREP = PROJECT / "outputs" / "prepared"

# Configuration
N_IMAGES = 5000          # encode this many products; raise to 20000 for medium preset
BACKBONE = "resnet50"    # "resnet50" or "clip"
BATCH = 32
TIMEOUT = 4              # seconds per image download

try:
    import torch
    DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
except Exception:
    DEVICE = "cpu"
print("Device:", DEVICE)

amz_path = PREP / "amazon_merged.parquet"
if not amz_path.exists():
    raise FileNotFoundError("Run RQ0 first to produce amazon_merged.parquet")
amz = pd.read_parquet(amz_path, columns=["parent_asin","images","rating_number"])
print(f"Amazon merged: {len(amz):,} rows")


Device: mps
Amazon merged: 300,000 rows


## 0b.1 Pick products to encode

We rank by `rating_number` (popularity) and take the top-N, de-duplicated by `parent_asin`.


In [2]:
amz["rating_number"] = pd.to_numeric(amz["rating_number"], errors="coerce").fillna(0)
amz = amz.drop_duplicates("parent_asin").sort_values("rating_number", ascending=False)
amz = amz.head(N_IMAGES).reset_index(drop=True)

def first_image_url(images):
    """Amazon 2023 images column: list of dicts, dict, str, ndarray, or NaN.
    Extract the first usable thumbnail URL."""
    # 1. Handle scalar NaN / None first
    if images is None: return None
    if isinstance(images, float) and np.isnan(images): return None
    # 2. Normalise to a Python list/dict regardless of source type
    if isinstance(images, np.ndarray):
        if images.size == 0: return None
        images = images.tolist()
    if isinstance(images, str):
        try:
            images = json.loads(images)
        except Exception:
            return None
    # 3. Empty after normalisation
    if not images: return None
    # 4. If it's a list, take first element
    if isinstance(images, list):
        if len(images) == 0: return None
        img = images[0]
    else:
        img = images
    # 5. Extract URL field
    if isinstance(img, dict):
        return img.get("thumb") or img.get("large") or img.get("hi_res")
    if isinstance(img, str):
        return img
    return None

amz["url"] = amz["images"].apply(first_image_url)
amz = amz.dropna(subset=["url"]).reset_index(drop=True)
print(f"Products with a usable image URL: {len(amz):,}")


Products with a usable image URL: 5,000


## 0b.2 Download + encode

We download each image and immediately feed it through the vision backbone. Images are not saved to disk. Failures are silently skipped.


In [3]:
from PIL import Image
import torch
from torchvision import transforms, models

# build the backbone
if BACKBONE == "resnet50":
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    # drop the classifier head -> 2048-dim global avg pool
    model.fc = torch.nn.Identity()
    EMB_DIM = 2048
    tfm = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
elif BACKBONE == "clip":
    from transformers import CLIPVisionModel, CLIPImageProcessor
    model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
    proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
    EMB_DIM = model.config.hidden_size
    tfm = None
else:
    raise ValueError(BACKBONE)

model = model.to(DEVICE).eval()
print(f"Backbone {BACKBONE} ready, emb_dim={EMB_DIM}")


Backbone resnet50 ready, emb_dim=2048


In [4]:
def fetch_image(url, timeout=TIMEOUT):
    try:
        req = Request(url, headers={"User-Agent":"Mozilla/5.0"})
        with urlopen(req, timeout=timeout) as r:
            data = r.read()
        return Image.open(io.BytesIO(data)).convert("RGB")
    except Exception:
        return None

def encode_batch(images):
    if BACKBONE == "resnet50":
        batch = torch.stack([tfm(im) for im in images]).to(DEVICE)
        with torch.no_grad():
            feats = model(batch)
        return feats.cpu().numpy()
    else:
        inputs = proc(images=images, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model(**inputs)
        return out.pooler_output.cpu().numpy()

embs = np.zeros((len(amz), EMB_DIM), dtype=np.float32)
ok_mask = np.zeros(len(amz), dtype=bool)
t0 = time.time()
buf_img, buf_idx = [], []
for i, url in enumerate(amz["url"].values):
    img = fetch_image(url)
    if img is not None:
        buf_img.append(img); buf_idx.append(i)
    if len(buf_img) >= BATCH or (i == len(amz)-1 and buf_img):
        feats = encode_batch(buf_img)
        for k, idx in enumerate(buf_idx):
            embs[idx] = feats[k]; ok_mask[idx] = True
        buf_img, buf_idx = [], []
    if (i+1) % 500 == 0:
        elapsed = time.time()-t0
        print(f"  {i+1:,}/{len(amz):,}  successes={ok_mask.sum():,}  elapsed={elapsed:.0f}s")
print(f"Done. {ok_mask.sum():,}/{len(amz):,} products encoded in {time.time()-t0:.0f}s")


  500/5,000  successes=480  elapsed=59s
  1,000/5,000  successes=992  elapsed=104s
  1,500/5,000  successes=1,472  elapsed=146s
  2,000/5,000  successes=1,984  elapsed=201s
  2,500/5,000  successes=2,496  elapsed=255s
  3,000/5,000  successes=2,976  elapsed=306s
  3,500/5,000  successes=3,488  elapsed=354s
  4,000/5,000  successes=4,000  elapsed=399s
  4,500/5,000  successes=4,480  elapsed=440s
  5,000/5,000  successes=5,000  elapsed=495s
Done. 5,000/5,000 products encoded in 495s


## 0b.3 Save


In [5]:
out = pd.DataFrame(embs[ok_mask], columns=[f"v{i:04d}" for i in range(EMB_DIM)])
out.insert(0, "parent_asin", amz["parent_asin"].values[ok_mask])
out_path = PREP / "amazon_product_image_embeddings.parquet"
out.to_parquet(out_path, index=False)
print(f"Saved -> {out_path}  shape={out.shape}  ({out_path.stat().st_size/1e6:.1f} MB)")


Saved -> /Users/bhanutejamalineni/Thesis/outputs/prepared/amazon_product_image_embeddings.parquet  shape=(5000, 2049)  (29.2 MB)


## ✅ Next
Now re-run `RQ2_graph_learning.ipynb`. It detects this file and concatenates image features with the text embeddings before initialising GNN item nodes — a true multimodal fusion.
